# UKB LOOK 1.0.0: Frozen-Classifier Incomplete-Modality Study

This notebook is the complete experiment interface after Steps 1-14 have produced a validated dataset. The ImageNet-pretrained classifier is trained only on complete CFP-OCT pairs, then frozen. Missing-modality filling, LOOK fitting, evaluation, statistics, and matrix analysis never fine-tune that classifier.


## 1. Configuration

Edit only the following cell, restart the kernel if `GPU_INDEX` changes, and run all cells. Every outer-axis list is expanded as a Cartesian product. A singleton list therefore runs exactly one choice. Candidate lists inside each LOOK profile are validation-search spaces; use separate named profiles with singleton candidate lists for controlled ablations.


In [ ]:
# A. Deployment and execution
from pathlib import Path
import os

PROJECT_ROOT = Path("/home/mengh/LOOK/2026_08_30_11_20_47")
DATA_ROOT = Path("/data/mengh/LOOK/2026_08_30_11_20_47")
DATASET_ROOT = DATA_ROOT / "dataset"
CACHE_ROOT = DATA_ROOT / "cache"
RUNS_ROOT = DATA_ROOT / "runs"
GPU_INDEX = 0
EXECUTION_MODE = "dry_run"  # "dry_run", "validation", or "test"
RESUME = True
RESTART = False
SMOKE_LIMIT = None
CHECK_ALL_IMAGE_PATHS = False
BOOTSTRAP_ITERATIONS = 2000
FROZEN_TEST_CONFIRMATION = ""  # set CONFIGURATION_FROZEN only after validation freeze

# B. Independent experiment axes: these lists form the outer Cartesian product
BACKBONES = ["resnet50"]  # LOOK 1.0.0 currently supports resnet50
FUSION_POSITIONS = ["input", "stem", "layer1", "layer2", "layer3", "layer4", "feature"]
SEEDS = [3407, 3408, 3409]
FILLING_STRATEGIES = ["normalized_mean", "paired_cgan"]

# C. Named complete-modality classifier profiles
CLASSIFIER_PROFILES = [{
    "name": "primary",
    "epochs": 50,
    "patience": 10,
    "effective_batch_size": 32,
    "micro_batch_size": 32,
    "num_workers": 8,
    "pretrained_lr": 1e-4,
    "new_layer_lr": 1e-3,
    "weight_decay": 1e-4,
    "warmup_epochs": 5,
    "sampler_power": 0.5,
    "amp": True,
}]

# D. Named independent paired-cGAN profiles; ignored by normalized_mean arms
GAN_PROFILES = [{
    "name": "primary",
    "gan_validation_fraction": 0.10,
    "gan_epochs": 100,
    "gan_patience": 10,
    "gan_batch_size": 16,
    "gan_num_workers": 8,
    "gan_learning_rate": 2e-4,
    "gan_beta1": 0.5,
    "gan_lambda_l1": 100.0,
    "gan_base_channels": 64,
}]

# E. Named LOOK/evaluation profiles. Use [4], [8], or [16] in separate profiles for fixed-factor ablations.
LOOK_PROFILES = [{
    "name": "primary",
    "enabled": True,
    "evaluate_random_missing": True,
    "missing_patterns": ["oct_missing", "cfp_missing"],
    "missing_ratios": [0.2, 0.4, 0.6, 0.8],
    "correction_nodes": ["all_available"],  # or explicit graph node names
    "downsample_factors": [4, 8, 16],
    "latent_dims": [16, 32, 64, 128, 256],
    "max_pca_rank": 256,
    "alpha_grid": [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0],
    "primary_metric": "macro_f1",
}]

os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_INDEX)


## 2. Import the installed release

Step 3 registers the project environment as `LOOK 1.0.0`. The notebook binds only that portable kernelspec name; each machine resolves it to its own project `.venv`.


In [ ]:
import json
import torch

from look_core.paths import ProjectPaths
from look_core.reproducibility import enforce_single_gpu
from look_core.study_grid import StudyGrid, expand_study_grid, run_study_grid

enforce_single_gpu()
if EXECUTION_MODE != "dry_run" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for experiment execution")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
PATHS = ProjectPaths.load(
    project_root=PROJECT_ROOT,
    data_root=DATA_ROOT,
    dataset_root=DATASET_ROOT,
    cache_root=CACHE_ROOT,
    runs_root=RUNS_ROOT,
)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU dry-run")


## 3. Resolve and validate the study plan

This cell validates every profile, the sealed-test guard, dataset contract, and deterministic experiment identity before expensive computation.


In [ ]:
GRID = StudyGrid(
    backbones=BACKBONES,
    fusion_positions=FUSION_POSITIONS,
    seeds=SEEDS,
    filling_strategies=FILLING_STRATEGIES,
    classifier_profiles=CLASSIFIER_PROFILES,
    gan_profiles=GAN_PROFILES,
    look_profiles=LOOK_PROFILES,
)
PHASE = "test" if EXECUTION_MODE == "test" else "validation"
CASES = expand_study_grid(
    GRID,
    PATHS,
    phase=PHASE,
    confirmation=FROZEN_TEST_CONFIRMATION,
    resume=RESUME,
    restart=RESTART,
    smoke_limit=SMOKE_LIMIT,
    check_all_image_paths=CHECK_ALL_IMAGE_PATHS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
print(f"Resolved configurations: {len(CASES)}")
for case in CASES[:10]:
    print(case.selection.experiment_id)
if len(CASES) > 10:
    print(f"... and {len(CASES) - 10} more")


## 4. Run or resume the complete experiment grid

Each configuration checks manifests and completion markers before reuse. Missing work resumes from stage state or epoch checkpoints; changed scientific configuration creates a new run ID.


In [ ]:
STUDY_RESULT = run_study_grid(
    GRID,
    PATHS,
    DEVICE,
    execute=EXECUTION_MODE != "dry_run",
    phase=PHASE,
    confirmation=FROZEN_TEST_CONFIRMATION,
    resume=RESUME,
    restart=RESTART,
    smoke_limit=SMOKE_LIMIT,
    check_all_image_paths=CHECK_ALL_IMAGE_PATHS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
print(json.dumps(STUDY_RESULT, indent=2))


## 5. Result location

The saved study plan records every resolved configuration and run ID. Completed experiment packages contain the frozen backbone identity, filling artifacts, validation/test metrics, predictions, LOOK search history, matrix spectra/eigenvectors, environment manifest, and implementation/data hashes.


In [ ]:
PLAN_DIR = RUNS_ROOT / "sweeps" / f"{PHASE}__{STUDY_RESULT['plan_id']}"
print("Study plan:", PLAN_DIR / "study_plan.json")
print("Progress:", PLAN_DIR / "progress.json")
print("Experiments:", RUNS_ROOT / "experiments")
